# Topic 10: Learning Curves: Watching Logistic Regression Learn

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 10 lecture note](https://dongchul.kim/csci3320-fall2026/topic/9c), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## 1. The loss curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler().fit(X_train)     # 30 means and 30 stds, from train only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):                                            # NEW: the cost from 9a
    p = np.clip(p, 1e-15, 1 - 1e-15)                           # NEW: see the box below
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))   # NEW

n, m = Z_train.shape
w, b = np.zeros(m), 0.0
alpha = 0.5
losses = []                                 # NEW: one number per step
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    losses.append(log_loss(p, y_train))     # NEW: the loss of the weights so far
    w -= alpha * (Z_train.T @ (p - y_train) / n)
    b -= alpha * np.mean(p - y_train)

for s in [0, 1, 10, 50, 100, 500, 999]:
    print(f"step {s:3d}: loss {losses[s]:.4f}")

plt.figure(figsize=(7, 4))                                     # NEW: the plot
plt.plot(losses, color="#2a78d6", lw=2)
plt.xlabel("step"); plt.ylabel("training loss (log loss)")
plt.title("The loss curve: 30 features, alpha = 0.5")
plt.show()

### When the curve goes up, or bounces

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
i = list(data.feature_names).index("mean radius")       # 9b's one-feature model
x_train = X_train[:, i]
mu, sigma = x_train.mean(), x_train.std()
z_train = (x_train - mu) / sigma

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

plt.figure(figsize=(7, 4))
for alpha, color in [(0.5, "#2a78d6"), (5, "#eb6834"), (30, "#1baf7a")]:
    w, b = 0.0, 0.0
    losses = []
    for step in range(1000):
        p = sigmoid(w * z_train + b)
        losses.append(log_loss(p, y_train))
        w -= alpha * np.mean((p - y_train) * z_train)
        b -= alpha * np.mean(p - y_train)
    print(f"alpha = {alpha:>4}:  step 20 {losses[20]:.4f}   step 100 {losses[100]:.4f}"
          f"   last four {np.round(losses[-4:], 4)}")
    plt.plot(losses[:100], color=color, lw=2, label=f"alpha = {alpha}")
plt.axhline(0.2954, color="#9ca3af", ls=":", lw=1)      # the bottom of the bowl, from 9b
plt.xlabel("step"); plt.ylabel("training loss")
plt.title("One feature, three learning rates (first 100 steps)")
plt.legend()
plt.show()

## 2. Training accuracy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

n, m = Z_train.shape
w, b = np.zeros(m), 0.0
alpha = 0.5
losses = []
train_acc = []                                                     # NEW
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    losses.append(log_loss(p, y_train))
    train_acc.append(((p >= 0.5).astype(int) == y_train).mean())   # NEW: grade the training data
    w -= alpha * (Z_train.T @ (p - y_train) / n)
    b -= alpha * np.mean(p - y_train)

for s in [0, 1, 5, 50, 100, 999]:
    print(f"step {s:3d}: loss {losses[s]:.4f}   training accuracy {train_acc[s]:.4f}"
          f"  ({round(train_acc[s] * n)} of {n})")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))                     # NEW: two panels
ax[0].plot(losses, color="#2a78d6", lw=2)
ax[0].set_title("training loss"); ax[0].set_ylabel("log loss")
ax[1].plot(train_acc, color="#2a78d6", lw=2)                       # NEW
ax[1].set_title("training accuracy, with the weights so far"); ax[1].set_ylabel("accuracy")
ax[1].set_ylim(0.88, 1.01)                        # NEW: zoom in; step 0 (0.36) is far below
for a in ax:
    a.set_xlabel("step")
plt.tight_layout()
plt.show()

## 3. The test set: accuracy and loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

n, m = Z_train.shape
w, b = np.zeros(m), 0.0
alpha = 0.5
losses, train_acc = [], []
test_losses, test_acc = [], []                                      # NEW
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    losses.append(log_loss(p, y_train))
    train_acc.append(((p >= 0.5).astype(int) == y_train).mean())
    p_test = sigmoid(Z_test @ w + b)                                # NEW: predict the test set
    test_losses.append(log_loss(p_test, y_test))                     # NEW: grade it,
    test_acc.append(((p_test >= 0.5).astype(int) == y_test).mean())  # NEW: nothing more
    w -= alpha * (Z_train.T @ (p - y_train) / n)                    # the update: training data only
    b -= alpha * np.mean(p - y_train)

print(" step  train loss  test loss  train acc  test acc")
for s in [0, 1, 20, 125, 500, 999]:
    print(f"{s:5d}    {losses[s]:.4f}     {test_losses[s]:.4f}     {train_acc[s]:.4f}    {test_acc[s]:.4f}")
low = int(np.argmin(test_losses))
print(f"\ntest loss is lowest at step {low}: {test_losses[low]:.4f}")
p_test = sigmoid(Z_test @ w + b)                  # the finished model, scored as in 9b
print(f"finished model on the test set: {((p_test >= 0.5).astype(int) == y_test).mean():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color="#2a78d6", lw=2, label=f"training ({len(y_train)} patients)")
ax[0].plot(test_losses, color="#eb6834", lw=2, ls="--", label="test (114 patients)")   # NEW
ax[0].set_title("loss"); ax[0].set_ylabel("log loss")
ax[0].set_ylim(0, 0.5)                            # NEW: zoom in; step 0 (0.69) is above
ax[1].plot(train_acc, color="#2a78d6", lw=2, label=f"training ({len(y_train)} patients)")
ax[1].plot(test_acc, color="#eb6834", lw=2, ls="--", label="test (114 patients)")      # NEW
ax[1].set_title("accuracy, with the weights so far"); ax[1].set_ylabel("accuracy")
ax[1].set_ylim(0.88, 1.01)
for a in ax:
    a.set_xlabel("step"); a.legend()                                                   # NEW: legends
plt.tight_layout()
plt.show()

## 5. Watching it happen: 30 training patients

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, y_train = X_train[:30], y_train[:30]      # NEW: only 30 training patients
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

n, m = Z_train.shape
w, b = np.zeros(m), 0.0
alpha = 0.5
losses, train_acc = [], []
test_losses, test_acc = [], []
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    losses.append(log_loss(p, y_train))
    train_acc.append(((p >= 0.5).astype(int) == y_train).mean())
    p_test = sigmoid(Z_test @ w + b)
    test_losses.append(log_loss(p_test, y_test))
    test_acc.append(((p_test >= 0.5).astype(int) == y_test).mean())
    w -= alpha * (Z_train.T @ (p - y_train) / n)
    b -= alpha * np.mean(p - y_train)

print(" step  train loss  test loss  train acc  test acc")
for s in [0, 3, 33, 54, 500, 999]:
    print(f"{s:5d}    {losses[s]:.4f}     {test_losses[s]:.4f}     {train_acc[s]:.4f}    {test_acc[s]:.4f}")
low = int(np.argmin(test_losses))
print(f"\ntest loss is lowest at step {low}: {test_losses[low]:.4f}")
print(f"training accuracy first reaches 1.0 at step {train_acc.index(1.0)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color="#2a78d6", lw=2, label=f"training ({len(y_train)} patients)")
ax[0].plot(test_losses, color="#eb6834", lw=2, ls="--", label="test (114 patients)")
ax[0].axvline(low, color="#6b7280", ls=":", lw=1.5, label=f"test loss lowest (step {low})")  # NEW
ax[0].set_title("loss"); ax[0].set_ylabel("log loss")
ax[0].set_ylim(0, 0.5)
ax[1].plot(train_acc, color="#2a78d6", lw=2, label=f"training ({len(y_train)} patients)")
ax[1].plot(test_acc, color="#eb6834", lw=2, ls="--", label="test (114 patients)")
ax[1].set_title("accuracy, with the weights so far"); ax[1].set_ylabel("accuracy")
ax[1].set_ylim(0.88, 1.01)
for a in ax:
    a.set_xlabel("step"); a.legend()
plt.tight_layout()
plt.show()

### Why the test loss goes back up

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, y_train = X_train[:30], y_train[:30]            # the same 30 patients
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train(steps):                                        # the same loop, as a function
    n, m = Z_train.shape
    w, b = np.zeros(m), 0.0
    for step in range(steps):
        p = sigmoid(Z_train @ w + b)
        w -= 0.5 * (Z_train.T @ (p - y_train) / n)
        b -= 0.5 * np.mean(p - y_train)
    return w, b

for steps in [54, 1000]:
    w, b = train(steps)
    p_train, p_test = sigmoid(Z_train @ w + b), sigmoid(Z_test @ w + b)
    truth_train = np.where(y_train == 1, p_train, 1 - p_train)   # probability given to the right answer
    truth_test = np.where(y_test == 1, p_test, 1 - p_test)
    wrong = truth_test < 0.5
    print(f"after {steps} steps: largest weight {np.abs(w).max():.2f}")
    print(f"  least sure training patient gives the truth {truth_train.min():.3f}")
    print(f"  {wrong.sum()} test patients wrong, giving the truth {truth_test[wrong].mean():.3f} on average")
    print(f"  test loss {-np.log(truth_test).mean():.4f} = right ones {-np.log(truth_test[~wrong]).sum() / 114:.4f}"
          f" + wrong ones {-np.log(truth_test[wrong]).sum() / 114:.4f}\n")

### Is it just this split?

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def experiment(seed, size):
    """Split with this seed, keep `size` training patients, run the loop for 1000 steps."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    X_train, y_train = X_train[:size], y_train[:size]
    scaler = StandardScaler().fit(X_train)
    Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)
    n, m = Z_train.shape
    w, b = np.zeros(m), 0.0
    test_losses = []
    for step in range(1000):
        p = sigmoid(Z_train @ w + b)
        test_losses.append(log_loss(sigmoid(Z_test @ w + b), y_test))
        w -= 0.5 * (Z_train.T @ (p - y_train) / n)
        b -= 0.5 * np.mean(p - y_train)
    train_acc = ((sigmoid(Z_train @ w + b) >= 0.5).astype(int) == y_train).mean()
    test_acc = ((sigmoid(Z_test @ w + b) >= 0.5).astype(int) == y_test).mean()
    turned_up = test_losses[-1] > 1.1 * min(test_losses)
    return train_acc, test_acc, turned_up

print("training   train acc   test acc   test acc over 50 splits   test loss")
print("patients    (mean)      (mean)     lowest     highest       turned up")
for size in [30, 100, 455]:
    r = np.array([experiment(seed, size) for seed in range(50)])
    print(f"{size:6d}      {r[:, 0].mean():.4f}     {r[:, 1].mean():.4f}     {r[:, 1].min():.4f}"
          f"     {r[:, 1].max():.4f}       {int(r[:, 2].sum())} of 50")

Back to the [Topic 10 lecture note](https://dongchul.kim/csci3320-fall2026/topic/9c).